In [0]:
%pip install databricks-feature-engineering

In [0]:
dbutils.library.restartPython()

In [0]:
from databricks.feature_engineering  import FeatureEngineeringClient
from pyspark.sql import functions as F


In [0]:
df_slv=spark.sql("select * from dev.ciencias_data.sesions_part1")
df_slv.display()

In [0]:
from pyspark.sql.types import StructType,StringType,StructField,IntegerType,ArrayType,LongType
schema=StructType([
    StructField("srcIp",StringType()),
    StructField("dstIp",StringType()),
    StructField("srcMac",StringType()),
    StructField("dstDataBytes",IntegerType()),
    StructField("srcDataBytes",IntegerType()),
    StructField("totDataBytes",IntegerType()),
    StructField("totBytes",IntegerType()),
    StructField("lastPacket",LongType()),
    StructField("firstPacket",LongType()),
    StructField("dstBytes",IntegerType()),
    StructField("srcBytes",IntegerType()),
    StructField("packetLen",ArrayType(IntegerType())),
    StructField("protocol",ArrayType(StringType()))
    ])
df_slv=df_slv.withColumn("data_struct",F.from_json(F.col("data"),schema))
"""
df_slv=df_slv.select(F.col("data_struct.dstDataBytes").alias("dst_data_bytes"),F.col("data_struct.dstBytes").alias("dst_bytes"),F.col("data_struct.srcIp"),F.col("data_struct.dstIp"),F.col("data_struct.srcMac"),F.col("data_struct.lastPacket").alias("last_packet"),F.col("data_struct.firstPacket").alias("first_packet"),F.col("data_struct.packetLen").alias("packet_len"))
df_final=df_slv.withColumn("session_duration",F.col("last_packet")-F.col("first_packet"))
df_final.display()
"""
slv_final=(df_slv.withColumn("src_ip",F.col("data_struct.srcIp"))
           .withColumn("dst_ip",F.col("data_struct.dstIp"))
           .withColumn("dst_data_bytes",F.col("data_struct.dstDataBytes"))
           .withColumn("tot_data_bytes",F.col("data_struct.totDataBytes"))
           .withColumn("last_packet",F.col("data_struct.lastPacket"))
           .withColumn("first_packet",F.col("data_struct.firstPacket"))
           .withColumn("packet_len",F.col("data_struct.packetLen"))
           .withColumn("protocol",F.col("data_struct.protocol"))).select("src_ip","dst_ip","dst_data_bytes","tot_data_bytes","last_packet","first_packet","packet_len","protocol")

slv_final.display()

In [0]:
%sql
use catalog dev;
create  schema if not exists silver;

In [0]:
slv_final.write.format("delta").mode("overwrite").saveAsTable("dev.silver.sessions")

In [0]:
df_feature=spark.read.table("dev.silver.sessions")

In [0]:
raw->bronze->silver->gold
              |
         feature store

In [0]:
df_feature = df_feature.withColumn("packetlen_min", F.array_min("packet_len")) \
             .withColumn("packetlen_max", F.array_max("packet_len"))

df_feature = df_feature.withColumn(
    "packetlen_media", 
    F.expr("aggregate(packet_len, 0D, (acumulador, x) -> acumulador + x) / size(packet_len)")
)
df_feature=df_feature.withColumn("session_duration",F.col("last_packet")-F.col("first_packet"))
df_feature.display()

#### MLOps

MLOps (Operaciones de Machine Learning) es un
conjunto de prácticas que combina el desarrollo (Dev) y las operaciones (Ops) para automatizar, gestionar y monitorear el ciclo de vida completo de los modelos de IA. Agiliza el despliegue, la formación continua y el mantenimiento de modelos, asegurando que sean fiables y eficientes en producción

El aprendizaje automático y la ciencia de datos dependen de los datos. El surgimiento de MLOps reabre interrogantes de larga data sobre la gestión y las operaciones de datos con una nueva urgencia. Por lo tanto, el frente más reciente en MLOps es el feature store.

Un feature store gestiona las "features" (características) o datos de entrada para un modelo de aprendizaje automático. A primera vista, no parece diferente de la gestión de datos en general; después de todo, las bases de datos con tablas de datos no son nada nuevo. En la práctica, el aprendizaje automático y la ciencia de datos comparten algunas necesidades (fiabilidad, linaje, versionado) con aplicaciones que consumen datos, a diferencia de otras que se centran en transacciones. El énfasis es distinto. Dada la importancia del machine learning, sus necesidades particulares (como requerir datos tanto en el momento del entrenamiento como en el de la inferencia) justifican una nueva clase de herramientas para la gestión de datos.
![image_1772675335965.png](./image_1772675335965.png "image_1772675335965.png")

### El problema con las features y el propósito de los Feature Stores

Los modelos de aprendizaje automático no consumen simplemente datos crudos. Lo que realmente tiene valor para un problema de aprendizaje son las funciones aplicadas a esos datos. En un modelo que predice la fuga de clientes (customer churn), por ejemplo, se podrían encontrar:
    Agregaciones de datos crudos en ventanas de tiempo, como las compras de los últimos 7 días.
    Combinaciones de conjuntos de datos, como la información demográfica del cliente unida a características de sus transacciones.
    Funciones complejas de la información del cliente, como el valor de vida estimado del cliente (Customer Lifetime Value).

El proceso de crear estos valores a partir de los datos se conoce como ingeniería de características (feature engineering).
##### DESCUBRIMIENTO (DISCOVERY)

Esas características pueden ser compartidas y reutilizadas en diferentes modelos, y puedes dar por hecho que necesitarán ser reutilizadas. Los equipos no pueden reutilizar lo que no pueden encontrar, por lo que uno de los propósitos principales de los feature stores es el descubrimiento: hacer visibles aquellas características que ya han sido refinadas de manera útil a partir de los datos crudos.

##### LINAGE (Linaje)

Compartir conlleva dependencias. Reutilizar una característica (feature) calculada para un propósito significa que cualquier cambio en su cálculo afectará ahora a muchos consumidores. Los productores de características deben comprender el linaje descendente (downstream lineage): ¿qué modelos y despliegues dependen de ella? De igual forma, los consumidores deben entender el linaje ascendente (upstream lineage) para usarla de forma fiable: ¿cómo se calcula y quién es el propietario?
##### SKEW (Sesgo o Desviación)

El problema del linaje no es nuevo, pero el ML presenta un reto distinto: gestionar la lógica de transformación. La ingeniería de características implica transformar datos crudos en dos contextos que pueden ser muy diferentes: el entrenamiento del modelo y su aplicación a nuevos datos (inferencia). Un modelo puede entrenarse en un entorno como Databricks, con computación distribuida y acceso a fuentes offline, pero ser desplegado en una aplicación web en Java que lo llama como un servicio.

In [0]:
from pyspark.sql.functions import udf
import uuid
def get_uuid():
    return str(uuid.uuid4())
uuid_v4=udf(get_uuid,StringType())
df_feature=df_feature.withColumn("id",uuid_v4())
df_feature.display()

In [0]:
%sql
use catalog dev;
create schema if not exists feature_store;

In [0]:
fe=FeatureEngineeringClient()
fe.create_table(
    name="dev.feature_store.sessions_arkime",
    primary_keys=["id"],
    df=df_feature,
    schema=df_feature.schema,
    description="features de sesiones de tráfico de red"
)

#### ¿Qué es una característica (feature)?

Para diseñar un feature store y su contenido, es fundamental entender qué representa una característica dentro de este paradigma. En esencia, se puede visualizar un feature store como una base de datos con funcionalidades adicionales integradas. Estos sistemas contienen tablas de características compuestas por filas y columnas tipadas que almacenan los valores de las mismas.

Aunque los feature stores pueden albergar datos no estructurados, adoptan un paradigma tabular, ya que la entrada directa de la mayoría de los modelos de aprendizaje automático es estructurada (incluso si se deriva de fuentes no estructuradas).
El concepto de Reutilización y claves primarias

Las características están diseñadas para ser reutilizadas al combinarse con otros conjuntos de datos. Para lograr esto, debe existir un mecanismo que identifique cómo unir las características, por lo cual las tablas de características necesitan una clave primaria (primary key). Los valores deben estar asociados a algo identificable de forma única; por ejemplo, un customer_id sería el candidato natural para una clave primaria en un modelo de clientes.
¿No es todo una característica?

¿Deberían gestionarse todos los datos de ciencia de datos y ML en un feature store? No. Un feature store aporta valor al gestionar datos transformados aptos para su uso directo en un modelo de machine learning, y no los datos crudos originales.

  Diferencia clave: Las transacciones no son características por sí mismas, pero las funciones derivadas de ellas sí lo son (por ejemplo, el total de llamadas a lo largo del tiempo).
  Valores derivados: Las características son, por definición, valores derivados de un proceso de transformación.

El paradigma del feature store se resume en: "crear la característica una vez, reutilizarla muchas". Los valores se calculan una sola vez, se almacenan, se gestionan y se comparten. Al calcular las características con antelación (ahead of time), se desacopla el cálculo de su uso, lo que permite ahorrar tiempo y costes significativos, especialmente en funciones computacionalmente costosas.